In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from grpo.utils import get_device, seed_everything

seed_everything(0)
get_device(), get_device("cpu")

(device(type='mps'), device(type='cpu'))

In [4]:
from grpo.data import parse_ground_truth, load_gsm8k

parse_ground_truth("blah\n#### 72")        # '72'
parse_ground_truth("blah\n#### 1,200")     # '1200'

train = load_gsm8k("train")
test = load_gsm8k("test")
train[0]["answer"]                         # '72'

'72'

In [5]:
import re
bad = [ex["answer"] for ex in train + test if not re.fullmatch(r"-?\d+(\.\d+)?", ex["answer"])]
len(bad), bad[:10]


(0, [])

In [6]:
from transformers import AutoTokenizer
from grpo.data import build_prompt

for name in ["HuggingFaceTB/SmolLM2-135M-Instruct", "Qwen/Qwen2.5-0.5B-Instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(build_prompt(tok, train[0]["question"]))
    print("=" * 60)


<|im_start|>system
Solve the math problem step by step. On the last line, write the final answer as '#### <number>'.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant

<|im_start|>system
Solve the math problem step by step. On the last line, write the final answer as '#### <number>'.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant



In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from grpo.data import build_prompt
from grpo.utils import get_device

name = "HuggingFaceTB/SmolLM2-135M-Instruct"
device = get_device()
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to(device)

model.eval()

prompt = build_prompt(tok, train[0]["question"])
inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to(device)

out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
completion = out[0,  inputs["input_ids"].shape[1]:]
print(tok.decode(completion))

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 492.14it/s]


Natalia sold 48 clips in April, and she sold half as many clips in May.

#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 48
#### 4


In [9]:
G = 4
out = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=True,
    temperature=1.0,
    top_p=1.0,
    top_k=0,
    repetition_penalty=1.0,
    num_return_sequences=G,
)
prompt_len = inputs["input_ids"].shape[1]
for i in range(G):
    print(f"--- sample {i} ---")
    print(tok.decode(out[i, prompt_len:]))


--- sample 0 ---
On April 1, Natalia sold 1 / 2 * 48 = 4 clips
On May 1, Natalia sold half as many clips as on April 1. So she sold 4 / (1 * 4) = 3 clips.
Natalia sold together 4 + 3 = 7 clips
#### 7
The answer is: 7<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im

In [10]:
completions = out[:, prompt_len:]
is_eos = completions == tok.eos_token_id
print(is_eos.int())
print(is_eos.int().argmax(dim=1))


tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
      

In [11]:
from grpo.sampling import sample_group

question = train[0]["question"]
completions, mask, truncated = sample_group(model, tok, question, G=4, max_new_tokens=256)

print(completions.shape, mask.sum(dim=1), truncated)
for i in range(2):
    text = tok.decode(completions[i][mask[i]], skip_special_tokens=True)
    print(f"--- sample {i} ---")
    print(text)


ValueError: too many values to unpack (expected 3)

In [23]:
question

'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'

In [ ]:
from grpo.reward import reward

question = train[0]["question"]
answer = train[0]["answer"]
input_ids, completions, mask, truncated = sample_group(model, tok, question, G=4, max_new_tokens=256)

texts = [tok.decode(completions[i][mask[i]], skip_special_tokens=True) for i in range(4)]
rewards = torch.tensor([reward(t, answer) for t in texts])
print(rewards)


tensor([0.0000, 0.1000, 0.1000, 0.1000])


In [25]:
texts

['Natalia sold clips to 48 people in April, and she sold half of those clips in May. So, Natalia sold approximately 48 * 5/2 = 245 clips in April.\n\nThe total number of clips sold by Natalia in both April and May is 5 people / clip sold per clip = 25 clips.\n\nNatalia sold 5/2 * 25 / 5 = 15 clips in April and 25 clips in May.\n#### 15\nThe answer is: 15',
 "First, let's determine the number of clips sold in April: Four8 canceling for 48.\nIn May, Natalia sold half as many clips: 36 canceling for 36.\n#### 36\n#### 48\n#### 48\n#### 46\n#### 54\n#### 65\n#### 70\n#### 78\n#### 85\n#### 94\n#### 100",
 'Natalia sold 48 clips to her friends, and in April, she sold 4 * 48 = 192 clips. She also sold half as many clips as in May, so she sold 192 / 2 = 98 clips in May.\n\nTo find the number of clips sold altogether in both April and May, we add the number of clips sold in April and the number of clips sold in May.\n\nC = V / (Q * (1 + R) * ((1 + R)) * S)\nC = 192 / (16 * (1 + 6)) * ((1 + 6) 

In [29]:
adv = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
adv

tensor([ 0.4990,  0.4990, -1.4970,  0.4990])

In [28]:
rewards

tensor([0.1000, 0.1000, 0.0000, 0.1000])

In [14]:
from grpo.sampling import sample_group, token_logprobs

prompt_ids, completions, mask, truncated = sample_group(model, tok, question, G=4, max_new_tokens=256)

with torch.no_grad():
    lp = token_logprobs(model, prompt_ids, completions)

print(lp.shape, completions.shape)
print((lp * mask).sum(dim=1) / mask.sum(dim=1))   # average log-prob per real token


torch.Size([4, 256]) torch.Size([4, 256])
tensor([-0.7134, -0.8930, -0.8708, -0.4448], device='mps:0')


In [15]:
from grpo.sampling import token_logprobs
with torch.no_grad():
    old_lp = token_logprobs(model, prompt_ids, completions)

new_lp = token_logprobs(model, prompt_ids, completions)

ratio = torch.exp(new_lp - old_lp)
print((ratio - 1)[mask].abs().max())

tensor(3.4332e-05, device='mps:0', grad_fn=<MaxBackward1>)


In [16]:
texts = [tok.decode(completions[i][mask[i]], skip_special_tokens=True) for i in range(4)]
rewards = torch.tensor([reward(t, answer) for t in texts], device=device)
adv = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
print(adv.shape)

eps = 0.2
ratio = torch.exp(new_lp - old_lp)
print(ratio.shape)
A = adv.unsqueeze(1)
print(A.shape)
surr = torch.min(ratio * A, torch.clamp(ratio, 1 - eps, 1 + eps) * A)

per_answer = (surr * mask).sum(dim=1) / mask.sum(dim=1)
loss = -per_answer.mean()
print(loss)


NameError: name 'reward' is not defined

In [ ]:
ref_model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to(device)
ref_model.eval()
ref_model.requires_grad_(False)

with torch.no_grad():
    ref_lp = token_logprobs(ref_model, prompt_ids, completions)

beta = 0.04
d = ref_lp - new_lp
kl = torch.exp(d) - d - 1

kl_per_answer = (kl * mask).sum(dim=1) / mask.sum(dim=1)
loss = -(per_answer - beta * kl_per_answer).mean()
print(kl_per_answer, loss)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 379.79it/s]


tensor([1.1462e-09, 6.9849e-10, 5.5705e-10, 4.0273e-09], device='mps:0',
       grad_fn=<DivBackward0>) tensor(1.1921e-07, device='mps:0', grad_fn=<NegBackward0>)


In [ ]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6)

optimizer.zero_grad()
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
model.eval()

with torch.no_grad():
    after_lp = token_logprobs(model, prompt_ids, completions)
ratio = torch.exp(after_lp - old_lp)
print("grad norm:", grad_norm.item())
print("max |ρ-1|:", (ratio - 1)[mask].abs().max().item())
print("avg log-prob change per answer:", ((after_lp - old_lp) * mask).sum(1) / mask.sum(1))
print("advantages:", adv)

grad norm: 1.8190854787826538
max |ρ-1|: 0.10447031259536743
avg log-prob change per answer: tensor([ 0.0033, -0.0072,  0.0091,  0.0076], device='mps:0')
advantages: tensor([ 0.4990, -1.4970,  0.4990,  0.4990], device='mps:0')


In [18]:
from grpo.sampling import token_logprobs

with torch.no_grad():
    new = token_logprobs(model, prompt_ids, completions)

    # old way: full log_softmax over the whole sequence
    P = prompt_ids.shape[1]
    full = torch.cat([prompt_ids.repeat(completions.shape[0], 1), completions], dim=1)
    logp = torch.log_softmax(model(full).logits[:, P - 1 : -1].float(), dim=-1)
    old = logp.gather(2, completions.unsqueeze(-1)).squeeze(-1)

print((new - old).abs().max())


tensor(2.9951e-06, device='mps:0')
